In [ ]:
# =============================================================================
#  Fermi Velocity Prediction Pipeline   — family + subfamily features
#  Input  : train_raw.csv, unique_m.csv, Fermi_169_unique_v5.csv
#  Output : UCI_with_Fermi_Velocity.csv
#           (21,263 rows | 81 features + material + family + subfamily + Predicted_vF)
# =============================================================================

import pandas as pd
import numpy as np
import re
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

# =============================================================================
# STEP 1 — Load data
# =============================================================================
df_train    = pd.read_csv("train_raw.csv")
df_unique_m = pd.read_csv("unique_m.csv")
df_fermi    = pd.read_csv("Fermi_169_unique_v5.csv")   # 162 unique materials

# Keep ORIGINAL case for regex-based classification; lowercase only as merge key
df_unique_m['material_lower'] = df_unique_m['material'].str.strip().str.lower()
df_unique_m['material_orig']  = df_unique_m['material'].str.strip()
df_fermi['material']          = df_fermi['material'].str.strip().str.lower()

# lookup: lowercase → original case
orig_case_map = dict(zip(df_unique_m['material_lower'], df_unique_m['material_orig']))


In [ ]:
## New version v4
import re

def get_subfamily(mat_orig):

    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    def has_explicit_coefficient_ge(element, string, threshold):
        m = re.search(rf"{element}(?![a-z])([0-9]+\.?[0-9]*)", string)
        if m is None:
            return False
        return float(m.group(1)) >= threshold

    def get_coeff(element, string):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return 0.0
        num = m.group(1)
        return 1.0 if num == '' else float(num)

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']

        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'

        if has('Bi', mat) and has('Sr', mat) and get_coeff('Bi', mat) > get_coeff('Hg', mat) and get_coeff('Bi', mat) > get_coeff('Tl', mat):
            return 'cuprate_BSCCO'

        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'

        BLOCKING_LAYER_METALS = ['V', 'Ti', 'Sc', 'Al', 'Mg', 'Cr']
        if has('As', mat) and has('O', mat) and any(has(m, mat) for m in BLOCKING_LAYER_METALS):
            return 'iron_other'

        if has('As', mat) and has('O', mat) and has('Te', mat):
            return 'iron_other'

        if has('As', mat) and (has('O', mat) or has('H', mat) or has('F', mat)):
            return 'iron_1111'

        if has_explicit_coefficient_ge('Se', mat, 2.5) and not has('As', mat):
            return 'iron_other'

        if any(has(a, mat) for a in ['Li', 'K', 'Rb', 'Cs', 'Tl', 'Na', 'Ca', 'Sr', 'Ba', 'Eu', 'Yb']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'

    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────

    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',  
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# vF LOOKUP TABLE -- direct-lookup subfamilies only.
#
# IMPORTANT: 'other', 'iron_other', and 'cuprate_other' are INTENTIONALLY
# ABSENT from this dictionary. These are the catch-all buckets for
# structurally ambiguous or genuinely mixed compounds (e.g. the Ca-Pt-As
# "10-3-8" phase inside iron_other, Hg+Bi hybrid cuprates inside
# cuprate_other, and chemically unrelated compounds like bismuthate
# perovskites or Fe-Ni-Zr Laves phases inside the generic 'other'). No
# single literature-backed Fermi velocity exists for these buckets, since
# they are deliberately heterogeneous by construction, not a real physical
# family. Per the actual training pipeline (vF_xi_Hc2.ipynb, Section 2:
# AMBIGUOUS_SUBFAMILIES / is_ambiguous_row), rows in these three subfamilies
# are ALWAYS routed to the trained ML fallback model (final_model.predict,
# followed by snap_to_valid) instead of a fixed lookup value.
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'iron_1111':                 152000,
    'iron_122':                  91981,
    'iron_111':                  45600,
    'iron_11':                    13700,
    'iron_245':                   79000,   
    'MgB2_type':                 560000,
    'conventional_A15':          212100,   
    'conventional_Nb_based':     726000,  
    'conventional_simple_metal': 1800000,
    'borocarbide':               276000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
}

def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

In [ ]:
 # Sanity check — runs when cell executes
_tests = [
    ('Mg0.95Fe0.05B2',          'MgB2_type',              560000),
    ('MgB2',                    'MgB2_type',              560000),
    ('CeIn3',                   'heavy_fermion_115',        7595),
    ('CeCoIn5',                 'heavy_fermion_115',        7595),
    ('CeCu2Si2',                'heavy_fermion_Ce',         7595),
    ('Tl1.4Hg0.6Ba2Ca2Cu3O',   'cuprate_Tl_based',       320000),
    ('HgBa2CuO4',               'cuprate_Hg_based',       290000),
    ('Ca0.5Na0.5Fe2As2',        'iron_122',               91981),
    ('Sn0.97As0.022Te0.978',    'chalcogenide',            95000),
    ('PbMo6S8',                 'chalcogenide',            95000),
    ('Nd1.47La0.4Ce0.13Cu1O4',  'cuprate_electron_doped', 230000),
    ('Nd1.85Ce0.15CuO4',        'cuprate_electron_doped', 230000),
    ('YBa2Cu3O7',               'cuprate_YBCO',           273400),
    ('La1.85Sr0.15CuO4',        'cuprate_LSCO',           245000),
    ('Bi2Sr2CaCu2O8',           'cuprate_BSCCO',          304000),
    ('Nb3Sn',                   'conventional_A15',       212100),
    ('FeSe',                    'iron_11',                 13700),
    ('LiFeAs',                  'iron_111',               45600),
    ('LuNi2B2C',                'borocarbide',            276000),
]
 
print("=== STEP 2 SANITY CHECK ===\n")
_bugs = 0
for _mat, _exp_sf, _exp_vf in _tests:
    _got_sf = get_subfamily(_mat)
    _got_vf = get_vF(_mat)
    _sf_ok  = 'OK ' if _got_sf == _exp_sf else 'BUG'
    _vf_ok  = 'OK ' if _got_vf == _exp_vf else 'BUG'
    if 'BUG' in (_sf_ok, _vf_ok):
        _bugs += 1
    print(f"  {_sf_ok} {_vf_ok} | {_mat:40} | {_got_sf:30} | vF: {_got_vf:>8,}")
 
print(f"\n  >>> {'ALL CLEAR — 0 bugs' if _bugs == 0 else f'{_bugs} BUGS REMAINING'}\n")
 


In [ ]:

# =============================================================================
# STEP 3 — Build df_combine: all 21,263 superconductors
# =============================================================================
df_combine = df_train.merge(
    df_unique_m[['material_lower', 'material_orig']],
    left_index=True, right_index=True,
    how='inner'
)
df_combine['subfamily'] = df_combine['material_orig'].apply(get_subfamily)
df_combine['family']    = df_combine['material_orig'].apply(get_family)
df_combine['low_confidence'] = df_combine['material_orig'].apply(is_low_confidence_extrapolation)  # FIX-9
# df_combine['low_confidence'] = df_combine['material_orig'].apply(is_ambiguous_row)  # FIX-9
df_combine['material']  = df_combine['material_lower']
df_combine.drop(columns=['material_lower', 'material_orig'], inplace=True)

print(f"df_combine shape : {df_combine.shape}")
df_combine.to_csv("UCI_family_subFamily.csv", index=False)
print("Saved: UCI_family_subFamily.csv")

print("\nFamily distribution (df_combine):")
print(df_combine['family'].value_counts().to_string())

print("\nSubfamily distribution (df_combine):")
print(df_combine['subfamily'].value_counts().to_string())

print(f"\nLow-confidence (extrapolated) rows: {df_combine['low_confidence'].sum()} ({df_combine['low_confidence'].mean():.2%})")


In [ ]:
# =============================================================================
# STEP 4 — Build df_169: the 169-sample training set with family + subfamily features
# =============================================================================
df_162 = df_fermi.copy()

# Add this at the end of STEP 4, after df_162 is built
df_162['material_orig'] = df_162['material'].map(orig_case_map)
df_162['vF (m/s)'] = df_162['material_orig'].apply(get_vF)   # ← reassign from VF_LOOKUP
df_162['subfamily'] = df_162['material_orig'].apply(get_subfamily)
df_162['family']    = df_162['material_orig'].apply(get_family)
df_162.drop(columns=['material_orig'], inplace=True)

print(f"\ndf_162 shape     : {df_162.shape}")
print(f"Unique materials : {df_162['material'].nunique()}")
print(f"Unique vF values : {df_162['vF (m/s)'].nunique()}")
print("\nSubfamily distribution (df_162):")
print(df_162['subfamily'].value_counts().to_string())
print(df_162.columns)

df_162.to_csv("Fermi_169_unique_v6.csv", index=False)

In [ ]:
import pandas as pd

# 1. Load datasets
fermi_file = "Fermi_169_unique_v6.csv"
uci_file = "UCI_family_subFamily.csv"

fermi_df = pd.read_csv(fermi_file)
uci_df = pd.read_csv(uci_file)

# 2. Specify the column name used to identify compounds in both files
# Replace 'material' with your actual column name (e.g., 'Compound', 'material_id')
compound_col = "material"

# Clean up any leading/trailing whitespace in compound names
fermi_df[compound_col] = fermi_df[compound_col].astype(str).str.strip()
uci_df[compound_col] = uci_df[compound_col].astype(str).str.strip()

# 3. Merge features with your fixed vf values
# This keeps all columns from Fermi dataset and attaches all 81 features from UCI dataset
merged_df = pd.merge(
    fermi_df,
    uci_df,
    on=compound_col,
    how="inner",
    suffixes=("", "_uci_dup"),
)

# Remove any duplicated columns if family/subfamily were present in both files
dup_cols = [col for col in merged_df.columns if col.endswith("_uci_dup")]
merged_df.drop(columns=dup_cols, inplace=True)

# 4. Save the complete dataset ready for training
output_file = "Fermi_169_complete_dataset.csv"
merged_df.to_csv(output_file, index=False)

print(f"Extraction complete!")
print(f"Total compounds matched: {len(merged_df)}")
print(f"Total features/columns: {merged_df.shape[1]}")

In [ ]:
import pandas as pd

df = pd.read_csv('Fermi_169_complete_dataset.csv')

print(f"Before dedup: {df.shape}")
print(f"Unique materials: {df['material'].nunique()}")

# Keep one row per material — mean Tc, first for everything else
df_162 = df.groupby('material', as_index=False).agg(
    {
        'critical_temp' : 'mean',   # average across duplicate Tc measurements
        'vF (m/s)'      : 'first',  # vF is identical across duplicates
        **{col: 'first' for col in df.columns 
           if col not in ['material', 'critical_temp', 'vF (m/s)']}
    }
)

print(f"\nAfter dedup: {df_162.shape}")
print(f"Unique materials: {df_162['material'].nunique()}")

# Sanity check — vF should have zero variance per material
assert df_162['material'].nunique() == len(df_162), "Duplicates still exist!"
assert df_162['vF (m/s)'].nunique() > 1, "vF looks wrong"

print(f"\nvF range: {df_162['vF (m/s)'].min():,.0f} – {df_162['vF (m/s)'].max():,.0f} m/s")
print(f"Tc range: {df_162['critical_temp'].min():.1f} – {df_162['critical_temp'].max():.1f} K")

df_162.to_csv('Fermi_169_unique_v8.csv', index=False)
print("\nSaved: Fermi_169_unique_v8.csv ✓")